# Daily Challenge: Fine-Tune a Question Answering Model with Haystack

Welcome to this comprehensive challenge where you'll learn to fine-tune a Question Answering (QA) model using Haystack's FARMReader on a SQuAD-style dataset.

## 🎯 Learning Objectives

By the end of this challenge, you will:
- Set up a complete Haystack environment for QA model training
- Understand SQuAD-style dataset format and preprocessing
- Fine-tune a pre-trained model on custom data
- Evaluate and compare model performance
- Create your own domain-specific QA system

## 📊 What is Haystack?

Haystack is an open-source NLP framework that helps you build intelligent search systems using Large Language Models (LLMs). It's perfect for:
- Document-based QA systems
- Retrieval-augmented generation (RAG) pipelines
- Fine-tuning transformer models
- Real-time feedback collection and model improvement

## 💡 SQuAD-Style Datasets

SQuAD (Stanford Question Answering Dataset) follows a specific structure:
- **Context**: A passage of text containing the answer
- **Question**: A natural language query about the context
- **Answer**: A text span that exists verbatim within the context
- **Start Index**: The character position where the answer begins

Let's get started! 🚀

## 1. Environment Setup and Dependencies

First, let's set up our environment with all the necessary dependencies for Haystack and model training.

In [ ]:
# Install Haystack with all necessary dependencies
!pip install farm-haystack[colab] torch transformers datasets

# Additional dependencies for visualization and data handling
!pip install matplotlib seaborn pandas numpy

# Check if GPU is available
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"Memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
else:
    print("⚠️ GPU not available. Training will be slower on CPU.")

In [ ]:
# Configure logging for better visibility during training
import logging
import os
import warnings

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Create directories for our work
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

print("✅ Environment setup complete!")
print("📁 Created directories: data/, models/, results/")

## 2. Configure Haystack and Initialize FARMReader

Now let's import Haystack components and initialize our FARMReader with a pre-trained model.

In [ ]:
# Import Haystack components
from haystack.nodes import FARMReader
from haystack.utils import launch_weaviate
from haystack import Document, Pipeline

# Optional: Configure telemetry (you can disable this if you prefer)
import haystack
# haystack.telemetry.disable_telemetry()  # Uncomment to disable

print("🔧 Haystack components imported successfully!")

# Initialize FARMReader with a pre-trained SQuAD model
print("🚀 Initializing FARMReader with pre-trained model...")

reader = FARMReader(
    model_name_or_path="deepset/roberta-base-squad2",
    use_gpu=torch.cuda.is_available(),
    context_window_size=150,
    return_no_answer=True,
    max_seq_len=384
)

print("✅ FARMReader initialized successfully!")
print(f"📋 Model: {reader.model_name_or_path}")
print(f"🖥️ Using GPU: {reader.use_gpu}")

## 3. Dataset Preparation and Loading

Let's download and prepare a SQuAD-style dataset for training. We'll use a subset of the SQuAD dataset for this tutorial.

In [ ]:
# Download and prepare training data
import json
import requests
from pathlib import Path

def download_squad_data():
    """Download a sample SQuAD dataset for training"""
    
    # Download SQuAD training data (subset)
    url = "https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v2.0.json"
    
    print("📥 Downloading SQuAD dataset...")
    
    try:
        response = requests.get(url)
        response.raise_for_status()
        
        with open("data/squad_train.json", "w", encoding="utf-8") as f:
            json.dump(response.json(), f, indent=2)
        
        print("✅ Dataset downloaded successfully!")
        return "data/squad_train.json"
    
    except Exception as e:
        print(f"❌ Error downloading dataset: {e}")
        print("📝 Creating a sample dataset instead...")
        
        # Create a small sample dataset if download fails
        sample_data = {
            "version": "2.0",
            "data": [
                {
                    "title": "Artificial Intelligence",
                    "paragraphs": [
                        {
                            "context": "Artificial intelligence (AI) is intelligence demonstrated by machines, in contrast to the natural intelligence displayed by humans and animals. Leading AI textbooks define the field as the study of intelligent agents: any device that perceives its environment and takes actions that maximize its chance of successfully achieving its goals. Colloquially, the term artificial intelligence is often used to describe machines that mimic cognitive functions that humans associate with the human mind, such as learning and problem solving.",
                            "qas": [
                                {
                                    "question": "What does AI stand for?",
                                    "id": "1",
                                    "answers": [
                                        {
                                            "text": "Artificial intelligence",
                                            "answer_start": 0
                                        }
                                    ],
                                    "is_impossible": False
                                },
                                {
                                    "question": "What do leading AI textbooks define the field as?",
                                    "id": "2", 
                                    "answers": [
                                        {
                                            "text": "the study of intelligent agents",
                                            "answer_start": 250
                                        }
                                    ],
                                    "is_impossible": False
                                }
                            ]
                        }
                    ]
                }
            ]
        }
        
        with open("data/squad_train.json", "w", encoding="utf-8") as f:
            json.dump(sample_data, f, indent=2)
        
        print("✅ Sample dataset created!")
        return "data/squad_train.json"

# Download the dataset
dataset_path = download_squad_data()

In [ ]:
# Explore and prepare the dataset
def explore_dataset(dataset_path):
    """Explore the structure of our SQuAD dataset"""
    
    with open(dataset_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    print("📊 Dataset Overview:")
    print(f"Version: {data['version']}")
    print(f"Number of articles: {len(data['data'])}")
    
    total_contexts = 0
    total_questions = 0
    
    for article in data['data']:
        total_contexts += len(article['paragraphs'])
        for paragraph in article['paragraphs']:
            total_questions += len(paragraph['qas'])
    
    print(f"Total contexts: {total_contexts}")
    print(f"Total questions: {total_questions}")
    
    # Show a sample
    print("\n📝 Sample Question-Answer Pair:")
    sample_article = data['data'][0]
    sample_paragraph = sample_article['paragraphs'][0]
    sample_qa = sample_paragraph['qas'][0]
    
    print(f"Context: {sample_paragraph['context'][:200]}...")
    print(f"Question: {sample_qa['question']}")
    if sample_qa['answers']:
        print(f"Answer: {sample_qa['answers'][0]['text']}")
        print(f"Answer start: {sample_qa['answers'][0]['answer_start']}")
    
    return data

# Explore the dataset
dataset = explore_dataset(dataset_path)

## 4. Fine-Tune the Model

Now let's fine-tune our FARMReader on the prepared dataset. We'll use a limited number of training samples for this demonstration.

In [ ]:
# Prepare data for training
def prepare_training_data(dataset, max_samples=100):
    """Convert SQuAD format to training format and limit samples for demo"""
    
    training_data = []
    count = 0
    
    for article in dataset['data']:
        for paragraph in article['paragraphs']:
            context = paragraph['context']
            
            for qa in paragraph['qas']:
                if count >= max_samples:
                    break
                    
                question = qa['question']
                
                # Handle cases with and without answers
                if qa['answers']:
                    answer_text = qa['answers'][0]['text']
                    answer_start = qa['answers'][0]['answer_start']
                    is_impossible = False
                else:
                    answer_text = ""
                    answer_start = 0
                    is_impossible = True
                
                training_data.append({
                    'context': context,
                    'question': question,
                    'answer_text': answer_text,
                    'answer_start': answer_start,
                    'is_impossible': is_impossible
                })
                
                count += 1
            
            if count >= max_samples:
                break
        if count >= max_samples:
            break
    
    print(f"📝 Prepared {len(training_data)} training samples")
    return training_data

# Prepare limited training data for demonstration
training_data = prepare_training_data(dataset, max_samples=50)

# Show sample training data structure
print("\n📋 Sample Training Data:")
sample = training_data[0]
for key, value in sample.items():
    if key == 'context':
        print(f"{key}: {value[:100]}...")
    else:
        print(f"{key}: {value}")

In [ ]:
# Fine-tune the model
print("🚀 Starting model fine-tuning...")
print("⏱️ This may take several minutes depending on your hardware...")

try:
    # Training parameters
    training_params = {
        "data_dir": "data",
        "train_filename": "squad_train.json",
        "n_epochs": 1,  # Using 1 epoch for demonstration
        "batch_size": 8,
        "learning_rate": 3e-5,
        "save_dir": "models/fine_tuned_reader"
    }
    
    print(f"📋 Training Parameters:")
    for key, value in training_params.items():
        print(f"  {key}: {value}")
    
    # Start training
    reader.train(
        data_dir=training_params["data_dir"],
        train_filename=training_params["train_filename"],
        use_gpu=torch.cuda.is_available(),
        n_epochs=training_params["n_epochs"],
        batch_size=training_params["batch_size"],
        learning_rate=training_params["learning_rate"],
        save_dir=training_params["save_dir"]
    )
    
    print("✅ Fine-tuning completed successfully!")
    
except Exception as e:
    print(f"❌ Training error: {e}")
    print("📝 Note: In case of errors, we'll proceed with the pre-trained model for demonstration")

## 5. Save and Load the Fine-Tuned Model

Let's save our fine-tuned model and demonstrate how to load it for future use.

In [ ]:
# Save the fine-tuned model
def save_model(reader, save_path):
    """Save the fine-tuned model"""
    try:
        reader.save(save_path)
        print(f"✅ Model saved to: {save_path}")
        return True
    except Exception as e:
        print(f"❌ Error saving model: {e}")
        return False

# Save our model
model_save_path = "models/my_fine_tuned_qa_model"
save_success = save_model(reader, model_save_path)

# Load the fine-tuned model (demonstration)
def load_fine_tuned_model(model_path):
    """Load a fine-tuned model"""
    try:
        fine_tuned_reader = FARMReader(
            model_name_or_path=model_path,
            use_gpu=torch.cuda.is_available(),
            context_window_size=150,
            return_no_answer=True
        )
        print(f"✅ Fine-tuned model loaded from: {model_path}")
        return fine_tuned_reader
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        print("📝 Using original pre-trained model instead")
        return reader

# Load the model (if saving was successful)
if save_success and os.path.exists(model_save_path):
    fine_tuned_reader = load_fine_tuned_model(model_save_path)
else:
    print("📝 Using the current reader instance for demonstration")
    fine_tuned_reader = reader

print("\n📋 Model Information:")
print(f"Model path: {fine_tuned_reader.model_name_or_path}")
print(f"Using GPU: {fine_tuned_reader.use_gpu}")
print(f"Context window: {fine_tuned_reader.context_window_size}")

## 6. Test Model Performance

Let's test our fine-tuned model by asking it questions and evaluating its performance.

In [ ]:
# Test the model with sample questions
def test_model_predictions(reader, test_cases):
    """Test the model with a set of questions and contexts"""
    
    results = []
    
    for i, test_case in enumerate(test_cases):
        context = test_case['context']
        question = test_case['question']
        expected_answer = test_case.get('expected_answer', 'Unknown')
        
        print(f"\n🔍 Test Case {i+1}:")
        print(f"Question: {question}")
        print(f"Expected: {expected_answer}")
        
        # Convert context to Document format
        docs = [Document(content=context)]
        
        # Get prediction
        try:
            prediction = reader.predict(query=question, documents=docs, top_k=1)
            
            if prediction['answers']:
                predicted_answer = prediction['answers'][0].answer
                confidence = prediction['answers'][0].score
                
                print(f"Predicted: {predicted_answer}")
                print(f"Confidence: {confidence:.3f}")
                
                results.append({
                    'question': question,
                    'expected': expected_answer,
                    'predicted': predicted_answer,
                    'confidence': confidence
                })
            else:
                print("Predicted: No answer found")
                results.append({
                    'question': question,
                    'expected': expected_answer,
                    'predicted': 'No answer',
                    'confidence': 0.0
                })
                
        except Exception as e:
            print(f"❌ Error in prediction: {e}")
    
    return results

# Create test cases
test_cases = [
    {
        'context': "Artificial intelligence (AI) is intelligence demonstrated by machines, in contrast to the natural intelligence displayed by humans and animals. Leading AI textbooks define the field as the study of intelligent agents: any device that perceives its environment and takes actions that maximize its chance of successfully achieving its goals.",
        'question': "What does AI stand for?",
        'expected_answer': "Artificial intelligence"
    },
    {
        'context': "Machine learning is a method of data analysis that automates analytical model building. It is a branch of artificial intelligence based on the idea that systems can learn from data, identify patterns and make decisions with minimal human intervention.",
        'question': "What is machine learning?",
        'expected_answer': "a method of data analysis that automates analytical model building"
    },
    {
        'context': "Deep learning is part of a broader family of machine learning methods based on artificial neural networks with representation learning. Learning can be supervised, semi-supervised or unsupervised.",
        'question': "What is deep learning based on?",
        'expected_answer': "artificial neural networks with representation learning"
    }
]

# Test our fine-tuned model
print("🧪 Testing Fine-Tuned Model:")
results = test_model_predictions(fine_tuned_reader, test_cases)

## 7. Model Evaluation and Comparison

Let's evaluate our model's performance and compare it with the base model to see improvements.

In [ ]:
# Compare with base model
print("🔄 Comparing with Base Model...")

# Initialize a fresh base model for comparison
base_reader = FARMReader(
    model_name_or_path="deepset/roberta-base-squad2",
    use_gpu=torch.cuda.is_available(),
    context_window_size=150,
    return_no_answer=True
)

print("🧪 Testing Base Model:")
base_results = test_model_predictions(base_reader, test_cases)

# Analyze results
import pandas as pd

def analyze_results(fine_tuned_results, base_results):
    """Compare results between fine-tuned and base models"""
    
    comparison_data = []
    
    for i, (ft_result, base_result) in enumerate(zip(fine_tuned_results, base_results)):
        comparison_data.append({
            'Test_Case': i + 1,
            'Question': ft_result['question'][:50] + "...",
            'Expected': ft_result['expected'],
            'Fine_Tuned_Answer': ft_result['predicted'],
            'Fine_Tuned_Confidence': ft_result['confidence'],
            'Base_Answer': base_result['predicted'],
            'Base_Confidence': base_result['confidence']
        })
    
    df = pd.DataFrame(comparison_data)
    return df

# Create comparison
if len(results) > 0 and len(base_results) > 0:
    comparison_df = analyze_results(results, base_results)
    
    print("\n📊 Performance Comparison:")
    print(comparison_df.to_string(index=False))
    
    # Calculate average confidences
    ft_avg_confidence = sum([r['confidence'] for r in results]) / len(results)
    base_avg_confidence = sum([r['confidence'] for r in base_results]) / len(base_results)
    
    print(f"\n📈 Average Confidence Scores:")
    print(f"Fine-tuned model: {ft_avg_confidence:.3f}")
    print(f"Base model: {base_avg_confidence:.3f}")
    
    if ft_avg_confidence > base_avg_confidence:
        print("✅ Fine-tuned model shows higher confidence!")
    else:
        print("📝 Base model maintains higher confidence")
else:
    print("⚠️ Unable to complete comparison due to missing results")

In [ ]:
# Visualize performance metrics
import matplotlib.pyplot as plt
import numpy as np

def visualize_performance(results, base_results):
    """Create visualizations of model performance"""
    
    if len(results) == 0 or len(base_results) == 0:
        print("⚠️ No results to visualize")
        return
    
    # Extract confidence scores
    ft_confidences = [r['confidence'] for r in results]
    base_confidences = [r['confidence'] for r in base_results]
    test_cases = [f"Q{i+1}" for i in range(len(results))]
    
    # Create comparison chart
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Confidence comparison
    x = np.arange(len(test_cases))
    width = 0.35
    
    ax1.bar(x - width/2, ft_confidences, width, label='Fine-tuned', alpha=0.8, color='skyblue')
    ax1.bar(x + width/2, base_confidences, width, label='Base Model', alpha=0.8, color='lightcoral')
    
    ax1.set_xlabel('Test Cases')
    ax1.set_ylabel('Confidence Score')
    ax1.set_title('Model Confidence Comparison')
    ax1.set_xticks(x)
    ax1.set_xticklabels(test_cases)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Average performance
    models = ['Fine-tuned', 'Base Model']
    avg_scores = [np.mean(ft_confidences), np.mean(base_confidences)]
    colors = ['skyblue', 'lightcoral']
    
    ax2.bar(models, avg_scores, color=colors, alpha=0.8)
    ax2.set_ylabel('Average Confidence')
    ax2.set_title('Average Performance Comparison')
    ax2.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for i, v in enumerate(avg_scores):
        ax2.text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    return fig

# Create visualizations
if len(results) > 0 and len(base_results) > 0:
    viz_fig = visualize_performance(results, base_results)
else:
    print("📊 Visualization skipped due to insufficient data")

## 8. Optional: Custom Dataset Creation

Now let's explore creating your own domain-specific SQuAD-style dataset and fine-tuning on it.

In [ ]:
# Create a custom domain-specific dataset
def create_custom_dataset(domain="technology"):
    """Create a custom SQuAD-style dataset for a specific domain"""
    
    # Sample domain-specific contexts and Q&As
    if domain == "technology":
        custom_data = {
            "version": "2.0",
            "data": [
                {
                    "title": "Machine Learning Fundamentals",
                    "paragraphs": [
                        {
                            "context": "Neural networks are computing systems inspired by biological neural networks. They consist of interconnected nodes called neurons that process information through weighted connections. Deep learning uses neural networks with multiple hidden layers to learn complex patterns in data. The backpropagation algorithm is commonly used to train these networks by adjusting weights based on prediction errors.",
                            "qas": [
                                {
                                    "question": "What are neural networks inspired by?",
                                    "id": "custom_1",
                                    "answers": [
                                        {
                                            "text": "biological neural networks",
                                            "answer_start": 67
                                        }
                                    ],
                                    "is_impossible": False
                                },
                                {
                                    "question": "What algorithm is commonly used to train neural networks?",
                                    "id": "custom_2",
                                    "answers": [
                                        {
                                            "text": "backpropagation algorithm",
                                            "answer_start": 298
                                        }
                                    ],
                                    "is_impossible": False
                                }
                            ]
                        }
                    ]
                },
                {
                    "title": "Cloud Computing",
                    "paragraphs": [
                        {
                            "context": "Cloud computing delivers computing services including servers, storage, databases, networking, software, analytics, and intelligence over the Internet. The three main service models are Infrastructure as a Service (IaaS), Platform as a Service (PaaS), and Software as a Service (SaaS). Cloud providers like Amazon Web Services, Microsoft Azure, and Google Cloud Platform offer scalable and cost-effective solutions.",
                            "qas": [
                                {
                                    "question": "What are the three main cloud service models?",
                                    "id": "custom_3",
                                    "answers": [
                                        {
                                            "text": "Infrastructure as a Service (IaaS), Platform as a Service (PaaS), and Software as a Service (SaaS)",
                                            "answer_start": 201
                                        }
                                    ],
                                    "is_impossible": False
                                }
                            ]
                        }
                    ]
                }
            ]
        }
    
    # Save custom dataset
    custom_dataset_path = f"data/custom_{domain}_dataset.json"
    with open(custom_dataset_path, 'w', encoding='utf-8') as f:
        json.dump(custom_data, f, indent=2)
    
    print(f"✅ Custom {domain} dataset created!")
    print(f"📁 Saved to: {custom_dataset_path}")
    
    return custom_dataset_path

# Create custom dataset
custom_dataset_path = create_custom_dataset("technology")

# Explore the custom dataset
custom_dataset = explore_dataset(custom_dataset_path)

In [ ]:
# Interactive Q&A with your fine-tuned model
def interactive_qa(reader):
    """Interactive question-answering session"""
    
    print("🤖 Interactive Q&A Session")
    print("You can ask questions about the training domain!")
    print("Type 'quit' to exit\n")
    
    # Sample contexts for demonstration
    sample_contexts = [
        "Artificial intelligence (AI) is intelligence demonstrated by machines, in contrast to the natural intelligence displayed by humans and animals. Leading AI textbooks define the field as the study of intelligent agents: any device that perceives its environment and takes actions that maximize its chance of successfully achieving its goals.",
        "Machine learning is a method of data analysis that automates analytical model building. It is a branch of artificial intelligence based on the idea that systems can learn from data, identify patterns and make decisions with minimal human intervention.",
        "Neural networks are computing systems inspired by biological neural networks. They consist of interconnected nodes called neurons that process information through weighted connections. Deep learning uses neural networks with multiple hidden layers to learn complex patterns in data."
    ]
    
    context = sample_contexts[0]  # Default context
    
    print(f"📄 Current Context: {context[:100]}...")
    print("=" * 50)
    
    # Simulate interactive session with predefined questions
    sample_questions = [
        "What does AI stand for?",
        "How do leading textbooks define AI?",
        "What is the goal of intelligent agents?"
    ]
    
    for question in sample_questions:
        print(f"\n❓ Question: {question}")
        
        try:
            docs = [Document(content=context)]
            prediction = reader.predict(query=question, documents=docs, top_k=1)
            
            if prediction['answers']:
                answer = prediction['answers'][0].answer
                confidence = prediction['answers'][0].score
                print(f"🤖 Answer: {answer}")
                print(f"📊 Confidence: {confidence:.3f}")
            else:
                print("🤖 Answer: I couldn't find an answer in the given context.")
                
        except Exception as e:
            print(f"❌ Error: {e}")
    
    print("\n✨ Interactive session completed!")

# Run interactive Q&A
interactive_qa(fine_tuned_reader)

## 🎉 Challenge Complete!

Congratulations! You've successfully completed the Haystack QA Fine-tuning Challenge. 

### 📋 What You've Accomplished:

1. ✅ **Environment Setup**: Configured Haystack with GPU support and proper logging
2. ✅ **Model Initialization**: Set up FARMReader with a pre-trained SQuAD model  
3. ✅ **Data Preparation**: Downloaded and processed SQuAD-style datasets
4. ✅ **Model Fine-tuning**: Trained the model on custom data
5. ✅ **Model Persistence**: Saved and loaded fine-tuned models
6. ✅ **Performance Testing**: Evaluated model predictions and confidence
7. ✅ **Comparative Analysis**: Compared fine-tuned vs. base model performance
8. ✅ **Custom Datasets**: Created domain-specific training data

### 🚀 Next Steps:

- **Experiment with longer training**: Try more epochs and larger datasets
- **Domain Specialization**: Create datasets for your specific use case
- **RAG Integration**: Combine with retrievers for full QA pipelines
- **Production Deployment**: Use Haystack pipelines in real applications
- **Advanced Techniques**: Explore few-shot learning and prompt engineering

### 📚 Additional Resources:

- [Haystack Documentation](https://docs.haystack.deepset.ai/)
- [SQuAD Dataset](https://rajpurkar.github.io/SQuAD-explorer/)
- [Hugging Face Models](https://huggingface.co/models)
- [FARM Framework](https://github.com/deepset-ai/FARM)

Keep experimenting and building amazing QA systems! 🎯